In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("prompt-evals-dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [8]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [9]:
with open("prompt-evals-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.333333333333333


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Extract AWS Region from S3 Bucket URL\n\nHere's a solution to extract the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_aws_region(s3_url):\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Supports formats:\n    - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n    - https://my-bucket.s3.us-west-2.amazonaws.com/key\n    - https://s3.us-west-2.amazonaws.com/my-bucket/key\n    - s3://my-bucket/key (returns 'us-east-1' as default)\n    \"\"\"\n    \n    # Method 1: Using regex for virtual-hosted-style URLs\n    match = re.search(r'\\.s3[.-]([a-z0-9-]+)\\.amazonaws\\.com', s3_url)\n    if match:\n        return match.group(1)\n    \n    # Method 2: Default region if no region is specified\n    if 's3.amazonaws.com' in s3_url or 's3://' in s3_url:\n        return 'us-east-1'\n    \n    return None\n\n\n# Test cases\ntest_urls = [\n    's3://my-bucket.s3.us-west-2.amazonaws.com/key',\n   